In [7]:
%load_ext autoreload
%autoreload 2

In [8]:
import os
import sys
from pathlib import Path

# Add project root to path so we can import the 'knowledge' package cleanly
# Assuming this notebook is inside a 'notebooks/' or root directory
project_root = Path(os.getcwd()).resolve().parents[1] # Adjust parents index if needed
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root added to path: {project_root}")

# Import your newly structured module
from kg_commit.knowledge.dataloader import CommitDataLoader

Project root added to path: E:\Projects\kgcommit


In [9]:
# 1. Map your project identifier to your local disk path
REPO_MAP = {
    "apache/groovy": r"E:\repos\groovy",
    "apache/hive": r"E:\repos\hive"
}

# 2. Grab a sample record (mirroring your CSV schema) for testing
sample_commit_id = "7b8480744ea6e6fb41efd4329bb470c8f3c763db"
sample_project = "apache/hive"

print("Configuration ready.")

Configuration ready.


In [10]:
# Initialize the loader with your path mappings
loader = CommitDataLoader(repo_map=REPO_MAP)

print(f"Attempting to fetch commit {sample_commit_id} from {sample_project}...\n")

# Run the fetch command
commit_payload = loader.fetch_commit_data(project=sample_project, commit_id=sample_commit_id)

# Verify the output structure
if commit_payload:
    print("✅ Success! Raw payload retrieved.")
    print("-" * 50)
    print(f"Commit ID: {commit_payload['commit_id']}")
    print(f"Author:    {commit_payload['author']}")
    print(f"Date:      {commit_payload['authored_datetime']}")
    print("-" * 50)
    print(f"Commit Message Snippet:\n{commit_payload['message'].strip()}")
    print("-" * 50)
    print(f"Diff Length: {len(commit_payload['diff'])} characters")
else:
    print("❌ Failed to fetch commit. Check your repo path or commit ID.")

Attempting to fetch commit 7b8480744ea6e6fb41efd4329bb470c8f3c763db from apache/hive...

Error fetching commit 7b8480744ea6e6fb41efd4329bb470c8f3c763db for apache/hive: SHA b'7b8480744ea6e6fb41efd4329bb470c8f3c763db' could not be resolved, git returned: b'7b8480744ea6e6fb41efd4329bb470c8f3c763db missing'
❌ Failed to fetch commit. Check your repo path or commit ID.


In [11]:
# Fetch a second time or check internal state
assert sample_project in loader._cached_repos, "Repo should be cached now!"
print(f"Cached instance verified: {loader._cached_repos[sample_project]}")

Cached instance verified: <git.repo.base.Repo 'E:\\repos\\hive\\.git'>


In [12]:
import os
from kg_commit.knowledge.dataloader import CommitDataLoader, JITDatasetAdapter

# Setup configurations
REPO_MAP = {"apache/groovy": r"E:\repos\groovy"}
CSV_PATH = r"E:\Projects\kgcommit\data\apachejit\projects\apache_groovy.csv"

# 1. Initialize our components
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)

# 2. Test reading records via the Adapter
print("--- Testing CSV Adapter Filtering ---")
records_stream = adapter.stream_records(CSV_PATH)

# Take the first two rows for validation
for i, record in enumerate(records_stream):
    if i >= 2: 
        break
    print(f"\nRecord #{i+1} parsed from CSV:")
    print(record)
    
    # 3. Use the filtered metadata to fetch Git info right away
    print(f"Fetching Git text data for commit: {record['commit_id']}...")
    git_payload = loader.fetch_commit_data(project=record['project'], commit_id=record['commit_id'])
    
    if git_payload:
        print(f"✅ Extracted Message length: {len(git_payload['message'])} chars")
        print(f"✅ Extracted Diff length: {len(git_payload['diff'])} chars")

--- Testing CSV Adapter Filtering ---

Record #1 parsed from CSV:
{'commit_id': '7b8480744ea6e6fb41efd4329bb470c8f3c763db', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1070355653'}
Fetching Git text data for commit: 7b8480744ea6e6fb41efd4329bb470c8f3c763db...
✅ Extracted Message length: 190 chars
✅ Extracted Diff length: 15148 chars

Record #2 parsed from CSV:
{'commit_id': '192b631e7be302ecde822546ba70a9853ddbda01', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1063298262'}
Fetching Git text data for commit: 192b631e7be302ecde822546ba70a9853ddbda01...
✅ Extracted Message length: 135 chars
✅ Extracted Diff length: 613 chars
